This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [ ]:
testable_data = data.get_testable_data("Example\\inputs\\case study 2 input-axial codes\\unclean version\\axial numerical.csv")
codes = data.get_codes("Example\\inputs\\case study 2 input-axial codes\\axial codes.csv")
all_scores = scores.get_BART_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
#make this go up to 10 instead of 88
all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Axial code(s)" column of testable_data to all_scores_expanded
all_scores_expanded["Axial code(s)"] = testable_data["Axial code(s)"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

Device set to use cuda:0


,Input phrase,1,2,3,4,5,6,7,8,9,10,Axial code(s)
0,I don't know. I'm balancing ideas against prizes.,0.818276,0.656613,0.923730,0.435810,0.751089,0.719943,0.532840,0.193599,0.197042,0.050770,8
1,"Before you shoot them down, say them out loud.",0.513787,0.358642,0.442446,0.673738,0.407870,0.343956,0.320399,0.204685,0.308389,0.505757,6
2,But it's all blockchain related so.,0.004536,0.666107,0.044405,0.102598,0.371961,0.001853,0.457222,0.003865,0.000345,0.000522,7
3,Do you want blockchain?,0.794437,0.433651,0.930096,0.203985,0.276549,0.167218,0.038395,0.024997,0.000564,0.001225,7
4,"No, I, no, I know I'm thinking of it because I...",0.796267,0.095206,0.059250,0.786745,0.189094,0.264405,0.155404,0.099765,0.089300,0.145688,1
...,...,...,...,...,...,...,...,...,...,...,...,...
236,Should I leave this here?,0.803624,0.181307,0.329306,0.625348,0.364933,0.606354,0.400493,0.249533,0.185207,0.317309,9
237,"Here we're all conquered and divided, right? Y...",0.031624,0.069762,0.015227,0.424032,0.627374,0.443111,0.618177,0.042833,0.666008,0.022596,5
238,"OK, like you gotta do like this spread. Leavin...",0.116454,0.682986,0.500865,0.382114,0.630031,0.476901,0.392401,0.087764,0.348811,0.102553,9
239,When are you going to go to sleep?,0.414435,0.075826,0.090305,0.511709,0.169080,0.069895,0.066521,0.007051,0.071137,0.026630,10


In [36]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 10 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]].max(axis=1) >= min) 
        & (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Axial code(s)"].tolist()
    predictions = all_scores_expanded_filtered[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    #make these labels also go to 88 instead of 11
    f1s = f1_score(ground_truths, predictions, labels=list(range(1, 89)), average=None, zero_division=0.0) 
    #mtx = confusion_matrix(ground_truths, predictions, labels=list(range(1, 89)))
    kappa = cohen_kappa_score(ground_truths, predictions, weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    return [f1, kappa]

In [ ]:
rows = []
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows.append({"min": i, "max": j, "kappa": results[1], "f1": results[0]})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_BART_axial_unclean_hackathon.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


,min,max,kappa,f1
0,0.95,1.00,0.115242,0.011932
1,0.90,0.95,0.023256,0.002841
2,0.85,0.90,-0.037838,0.000000
3,0.80,0.85,0.029412,0.008207
4,0.75,0.80,0.125000,0.013880
5,0.70,0.75,0.028698,0.003535
6,0.65,0.70,0.021359,0.006818
7,0.60,0.65,0.047337,0.007071
8,0.55,0.60,0.093023,0.007576
9,0.50,0.55,0.112676,0.017045
